# 03 · Je eigen server inrichten

Tot nu toe bewaar je de databank op je laptop en in Codespaces. Om straks gegevens met een partner uit te wisselen, richt je een computer in die via internet bereikbaar blijft. Die computer huurt je team bij DigitalOcean. Je installeert er een MQTT-broker op: een programma dat berichten van verzenders doorgeeft aan aangemelde ontvangers.

Je maakt een eigen server aan, meldt je aan met een wachtwoord en installeert de benodigde programma's. Vervolgens test je vanuit Codespaces of een bericht veilig heen en terug kan. Je controleert ook de beveiligde webverbinding waarop later de gegevens-API komt.

**Nodig:** je bestaande Codespace, je instellingen uit notebook 01, een aanvaarde uitnodiging voor het DigitalOcean-team **lanupro-tech** en toegang tot het project **starten-met-iot**. Je hebt geen Python-installatie op je laptop nodig.

Voer de codecellen in volgorde uit en onderbreek bij de handmatige stappen. De opdrachten in grijze tekstvakken met het label **bash** zijn voor een terminal. Boven ieder blok staat of je in Codespaces of op de server werkt.

<div style="background-color:#fff8e1;color:#6d4c00;border-left:5px solid #f9a825;padding:14px;margin:12px 0;border-radius:4px;"><strong>Voor je begint</strong><br>Gebruik team <strong>lanupro-tech</strong> en project <strong>starten-met-iot</strong>. Maak één eigen Droplet met Debian 13 en het goedkoopste Basic/Regular-plan. Kies wachtwoordaanmelding en laat Startup scripts uitgeschakeld. Hieronder loop je alle keuzes afzonderlijk door.</div>

## 1. Je eigen werkplek ophalen

**Uitvoeren:** in Codespaces (Notebook 03)

Open `notebooks/03_eigen_server_inrichten.ipynb` en selecteer dezelfde Python-kernel als in notebook 01. Je deelnemerscode bepaalt straks het gedeelte van de broker waarop jij berichten mag plaatsen. Daarom gebruiken we opnieuw je opgeslagen instellingen.

In [1]:
# TIJDELIJKE DOCENTCEL: verdwijnt door de tag oplossing uit de studentenversie.
# Een bestaande geldige student/-branch blijft actief.
_DOCENT_BRANCH_OVERRIDE = "student/proefdocent"

In [2]:
from pathlib import Path
import subprocess
import json
import re
import sys
import ipaddress

resultaat = subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True)
if resultaat.returncode:
    raise RuntimeError("Open notebook 03 vanuit de practicumrepository in Codespaces.")
hoofdmap = Path(resultaat.stdout.strip())
branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=hoofdmap, text=True).strip()
if not re.fullmatch(r"student/[a-z0-9][a-z0-9_-]*", branch) and "_DOCENT_BRANCH_OVERRIDE" in globals():
    branch = _DOCENT_BRANCH_OVERRIDE
if not re.fullmatch(r"student/[a-z0-9][a-z0-9_-]*", branch):
    raise RuntimeError("Selecteer je eigen student/voornaam_achternaam-branch uit notebook 01.")
instellingenpad = hoofdmap / "gegevens/inlevering/01_werkplekcontrole.json"
if not instellingenpad.is_file():
    raise FileNotFoundError("Bewaar eerst de werkplekcontrole uit notebook 01.")
instellingen = json.loads(instellingenpad.read_text(encoding="utf-8"))
deelnemer = instellingen.get("deelnemerscode", "")
if instellingen.get("branch") != branch or deelnemer != branch.removeprefix("student/"):
    raise ValueError("Je opgeslagen instellingen horen niet bij deze branch. Controleer notebook 01.")
if not re.fullmatch(r"[a-z0-9][a-z0-9_-]{0,62}", deelnemer):
    raise ValueError("Gebruik een deelnemerscode van maximaal 63 tekens, met kleine letters, cijfers, _ of -.")
werkmap = hoofdmap / "gegevens/werk"
werkmap.mkdir(parents=True, exist_ok=True)
print("Deelnemerscode:", deelnemer)
print("Servernaam:", "iot-" + deelnemer.replace("_", "-"))
print("Team: lanupro-tech | Project: starten-met-iot")

Deelnemerscode: proef
Servernaam: iot-proef
Team: lanupro-tech | Project: starten-met-iot


## 2. Wat huur je bij DigitalOcean?

Een **server** is een computer die diensten levert aan andere computers. Bij DigitalOcean heet een virtuele server een **Droplet**. Hij beschikt over rekentijd, werkgeheugen en een schijf, maar je bedient hem op afstand. Je laptop hoeft dus niet aan te blijven om de server bereikbaar te houden.

**Debian** is het besturingssysteem op die server. Het beheert bestanden, gebruikers en programma's. Ubuntu en Fedora zijn andere Linux-distributies: ze vervullen dezelfde hoofdrol, maar verschillen onder meer in softwareversies en beheeropdrachten. Hier gebruikt iedereen Debian 13, zodat de installatieopdrachten overeenkomen.

De **hostname** is de herkenbare naam van je server. Het **publieke IPv4-adres** is het internetadres waarmee Codespaces hem bereikt. Een **poort** geeft aan welk programma een verbinding moet ontvangen. Op dezelfde server gebruiken we poort 22 voor beheer, 8883 voor beveiligd MQTT en 443 voor HTTPS.

Je eigen laptop en de server zijn allebei computers. Het verschil voor deze oefening is de bereikbaarheid: een campusnetwerk kan verbindingen tussen laptops blokkeren. De notebooks draaien in Codespaces en leggen van daaruit verbinding met DigitalOcean. Je campusverbinding gebruik je om Codespaces te bedienen. Controleer die toegang vooraf; geen enkele opzet garandeert toegang vanuit ieder netwerk.

**Vraag 1.** Waar draait Python wanneer je een notebookcel uitvoert, en waar draait Mosquitto na deze installatie? Wat gebeurt er met Mosquitto als je je laptop afsluit?

**Jouw antwoord (Vraag 1):**

Voeg hier je antwoord in. Benoem de computers afzonderlijk.

**Antwoord 1**

De notebookcode draait in Codespaces bij GitHub. Mosquitto draait op de eigen Droplet bij DigitalOcean. Als de Droplet blijft draaien, blijft Mosquitto beschikbaar wanneer de laptop sluit. Er worden dan alleen geen nieuwe berichten verstuurd door een gestopt notebook.

## 3. Het juiste team en project kiezen

**Uitvoeren:** op de DigitalOcean-website

Een DigitalOcean-account is je persoonlijke aanmelding. Een **team** bepaalt wie cloudresources mag beheren en waar de kosten terechtkomen. Een **project** groepeert resources binnen dat team. Een projectnaam maakt resources niet automatisch privé voor één teamlid.

1. Aanvaard de uitnodiging voor **lanupro-tech** met het daarvoor opgegeven account. Je kunt GitHub als aanmeldmethode gebruiken; de teamuitnodiging blijft een afzonderlijke stap.
2. Open [cloud.digitalocean.com](https://cloud.digitalocean.com/). Kies via je profiel **Switch Teams** en selecteer **lanupro-tech**.
3. Open het project **starten-met-iot**. Controleer de team- en projectnaam voordat je doorgaat.
4. Ontbreekt het team, project of de aanmaakknop, laat je toegangsrechten controleren. Maak daarvoor geen tweede persoonlijk betaalaccount of vervangend project aan.

Je werkt alleen aan jouw eigen Droplet. Gebruik de server van een partner later uitsluitend via het daarvoor gegeven MQTT-leesaccount. De docent bepaalt de teamrechten en de facturering. Zie [DigitalOcean: teamlidmaatschap](https://docs.digitalocean.com/platform/teams/how-to/manage-membership/).

## 4. Eén eigen Droplet aanmaken

**Uitvoeren:** op de DigitalOcean-website

Klik op **Create → Droplet**. Je kiest nu waar de server staat, hoeveel geheugen hij krijgt en hoe je aanmeldt. Doorloop de onderdelen hieronder in de volgorde van het formulier; de precieze plaats kan door een schermupdate veranderen.

Je maakt pas een server aan bij de laatste knop. Je kunt de keuzes daarvoor rustig aanpassen. De screenshots tonen waar de onderdelen staan. Neem de voorbeeldwaarden in de afbeeldingen niet over: bij ieder onderdeel staat jouw keuze.

### 4.1 Regio en netwerk binnen het datacenter

**Kies: Amsterdam**, als het goedkoopste plan daar beschikbaar is. De regio is de plaats van het datacenter. Een locatie dichtbij België houdt de afstand tot je server beperkt.

Open de regiokeuzelijst en selecteer Amsterdam. Zie je meerdere datacenters bij dezelfde plaats, gebruik dan de standaard aangeboden keuze. Staat de regio al op New York zoals in het voorbeeld, wijzig die.

**VPC network: behoud het standaardnetwerk.** Dit is het interne netwerk tussen cloudresources in dezelfde regio. Je hoeft er geen eigen netwerk voor aan te maken.

![De regiokeuzelijst en het standaard VPC-netwerk; het bronvoorbeeld staat op New York.](../../afbeeldingen/schermafbeeldingen/03_regio.png)

*Figuur 1. De regiokeuzelijst en het standaard VPC-netwerk; het bronvoorbeeld staat op New York. Officiële DigitalOcean-bronafbeelding, gecontroleerd op 17 november 2026. De keuzes voor deze oefening staan in de tekst erboven. [Bron en rechten](../../afbeeldingen/schermafbeeldingen/BRONNEN.md).*

### 4.2 Besturingssysteem

**Kies: OS → Debian → 13 x64.** Het besturingssysteem beheert de server en voert je programma's uit. Door allemaal Debian 13 te kiezen, kun je dezelfde opdrachten gebruiken.

Selecteer de regel **Debian**. Controleer rechts de versie **13** en de aanduiding **x64**. Die laatste betekent dat de server een 64-bits besturingssysteem gebruikt. Laat de andere tabbladen, zoals Solutions en Custom Images, ongebruikt: je installeert de benodigde programma's later zelf.

![De beschikbare besturingssystemen. Klik op Debian 13; Ubuntu is de selectie in het bronvoorbeeld.](../../afbeeldingen/schermafbeeldingen/03_besturingssysteem.png)

*Figuur 2. De beschikbare besturingssystemen. Klik op Debian 13; Ubuntu is de selectie in het bronvoorbeeld. Officiële DigitalOcean-bronafbeelding, gecontroleerd op 17 november 2026. De keuzes voor deze oefening staan in de tekst erboven. [Bron en rechten](../../afbeeldingen/schermafbeeldingen/BRONNEN.md).*

### 4.3 Het goedkoopste serverplan

**Kies: Bundled → Basic → Regular → het goedkoopste plan.**

1. Verschijnt een keuze tussen **Bundled** en **v5**, kies dan **Bundled**. Dit groepeert processor, geheugen en schijf in een vast plan.
2. Kies **Basic** bij de plannen. Hierbij deel je de beschikbare processortijd met andere virtuele servers. Voor deze oefening starten we met de kleinste configuratie.
3. Kies **Regular** bij CPU Options.
4. Klik zo nodig op **Show all plans** en kies de goedkoopste kaart. De gepubliceerde laagste prijs op **17 november 2026** is **$4 per maand**, met **512 MiB RAM, 1 vCPU en 10 GiB SSD**. Controleer het bedrag in jouw formulier.

| Aanduiding | Wat betekent dit? |
|---|---|
| 1 vCPU | De virtuele processor voert opdrachten uit. |
| 512 MiB RAM | Het werkgeheugen voor actieve programma's. |
| 10 GiB SSD | De schijfruimte voor Debian, software en databestanden. |
| 500 GiB transfer | Het in dit plan opgenomen uitgaande dataverkeer. |

De prijs is de gepubliceerde basisprijs; eventuele belastingen en extra verbruik kunnen daarbovenop komen. Met vier servers is de basisprijs dus **$16 per maand**. [Actuele prijzen](https://www.digitalocean.com/pricing/droplets), [uitleg over serverplannen](https://docs.digitalocean.com/products/droplets/concepts/choosing-a-plan/).

Zie je de $4-optie niet, controleer eerst **Regular** en **Show all plans**. Als het plan in deze regio niet beschikbaar is, laat de docent een regio aanwijzen waar het wel beschikbaar is. Kies niet ongemerkt een duurdere kaart.

![De knoppen Basic, Regular en Show all plans. Het bronvoorbeeld toont Premium Intel en een duurder plan; selecteer zelf Regular en de laagste prijs.](../../afbeeldingen/schermafbeeldingen/03_serverplan.png)

*Figuur 3. De knoppen Basic, Regular en Show all plans. Het bronvoorbeeld toont Premium Intel en een duurder plan; selecteer zelf Regular en de laagste prijs. Officiële DigitalOcean-bronafbeelding, gecontroleerd op 17 november 2026. De keuzes voor deze oefening staan in de tekst erboven. [Bron en rechten](../../afbeeldingen/schermafbeeldingen/BRONNEN.md).*

### 4.4 Extra opslag en back-ups

**Additional Storage / Add Volume: uit laten.** Je serverplan bevat al een schijf. Een volume is een extra gehuurde schijf met een afzonderlijke prijs. Heb je per ongeluk het grote configuratievak hieronder geopend, haal dan het vinkje bij **Add additional storage** weg. De extra vragen over grootte, formatteren en naam hoef je dan niet in te vullen.

![Het uitgeklapte vak voor extra opslag. Laat Add additional storage voor dit practicum uitgeschakeld.](../../afbeeldingen/schermafbeeldingen/03_extra_opslag.png)

*Figuur 4. Het uitgeklapte vak voor extra opslag. Laat Add additional storage voor dit practicum uitgeschakeld. Officiële DigitalOcean-bronafbeelding, gecontroleerd op 17 november 2026. De keuzes voor deze oefening staan in de tekst erboven. [Bron en rechten](../../afbeeldingen/schermafbeeldingen/BRONNEN.md).*

**Enable automated backups: uit laten.** Deze betaalde optie maakt automatisch kopieën van de Droplet. Voor dit practicum bewaar je de lesbestanden en resultaten volgens de notebookinstructies. Door deze optie uit te laten, betaal je alleen voor het gekozen serverplan. Zie [DigitalOcean: back-ups](https://docs.digitalocean.com/products/backups/).

### 4.5 Aanmelden met een wachtwoord

**Kies: Password.** Daarmee stel je het eerste beheerderswachtwoord van de server in. Je maakt in deze oefening geen eigen SSH-sleutelpaar en hoeft geen publieke sleutel te uploaden.

1. Klik in het onderdeel **Authentication** op **Password**.
2. Maak een uniek sterk wachtwoord dat aan de getoonde voorwaarden voldoet. Gebruik bij voorkeur een willekeurig wachtwoord uit je wachtwoordmanager.
3. Bewaar het als het **root-wachtwoord van jouw Droplet**, samen met de servernaam. Zet het niet in een notebook of GitHub-bestand.

Je gebruikt later wel de opdracht `ssh` om versleuteld met dit wachtwoord aan te melden. **SSH is de verbinding; Password is de aanmeldmethode.** Bij een nieuwe Codespace blijven het serveraccount en het wachtwoord bestaan. Je maakt dan opnieuw verbinding en controleert opnieuw de servervingerafdruk; een gebruikerssleutel genereren is niet nodig.

![De tabbladen SSH Keys en Password. Klik op Password; het bronvoorbeeld toont nog het andere tabblad.](../../afbeeldingen/schermafbeeldingen/03_aanmeldmethode.png)

*Figuur 5. De tabbladen SSH Keys en Password. Klik op Password; het bronvoorbeeld toont nog het andere tabblad. Officiële DigitalOcean-bronafbeelding, gecontroleerd op 17 november 2026. De keuzes voor deze oefening staan in de tekst erboven. [Bron en rechten](../../afbeeldingen/schermafbeeldingen/BRONNEN.md).*

Meer informatie: [DigitalOcean: aanmelden via SSH met een wachtwoord of sleutel](https://docs.digitalocean.com/products/droplets/how-to/connect-with-ssh/).

### 4.6 Netwerk, monitoring en startup scripts

Stel de volgende opties in. Open **Additional Options** als dat onderdeel dichtgeklapt is.

| Optie | Jouw keuze | Korte uitleg |
|---|---|---|
| Public IPv4 address | **Aan** | Dit internetadres gebruik je vanuit Codespaces. |
| Public IPv6 address | **Uit** | IPv6 is een ander adresformaat; deze handleiding gebruikt IPv4. |
| Improved Metrics and monitoring | **Uit** | Dit zou extra meetsoftware installeren om het servergebruik te volgen. We houden de eerste inrichting eenvoudig. |
| Startup scripts | **Uit; tekstvak leeg** | Dit kan tijdens de eerste opstart automatisch installatieopdrachten uitvoeren. Je voert de installatie straks zelf stap voor stap uit. |
| Managed Database | **Uit** | Dit is een afzonderlijke betaalde databasedienst. Je gebruikt straks SQLite op de schijf van je Droplet. |

Bij **Startup scripts** hoef je dus niets te schrijven of te plakken. Deze instelling voor de eerste inrichting staat los van het later starten van een reeds geïnstalleerde broker. [Uitleg over startup scripts](https://docs.digitalocean.com/products/droplets/how-to/provide-user-data/).

![Netwerk en aanvullende opties. Houd alleen Public IPv4 aan; zet ook de in dit bronvoorbeeld aangevinkte monitoring uit.](../../afbeeldingen/schermafbeeldingen/03_netwerk_extra_opties.png)

*Figuur 6. Netwerk en aanvullende opties. Houd alleen Public IPv4 aan; zet ook de in dit bronvoorbeeld aangevinkte monitoring uit. Officiële DigitalOcean-bronafbeelding, gecontroleerd op 17 november 2026. De keuzes voor deze oefening staan in de tekst erboven. [Bron en rechten](../../afbeeldingen/schermafbeeldingen/BRONNEN.md).*

### 4.7 Aantal, servernaam, project en jouw voornaam als tag

| Optie | Wat vul je in? | Waarom? |
|---|---|---|
| Quantity | **1** | Je maakt één server voor jezelf. |
| Hostname / Give your Droplet a name | De naam uit de eerste codecel, bijvoorbeeld `iot-jan-janssens` | Hiermee herken je jouw server in de lijst en in de terminal. |
| Select a project | **starten-met-iot** | Dit houdt de practicumservers samen in het afgesproken project. |
| Tags | **Jouw voornaam**, bijvoorbeeld `jan` | Zo kan de docent de servers eenvoudig terugvinden en filteren. |

Klik bij **Tags** op **Enter tag name**, typ je voornaam en druk op Enter. Controleer dat je naam als afzonderlijk label zichtbaar wordt. Gebruik kleine letters en bij een samengestelde voornaam een koppelteken, bijvoorbeeld `jan-pieter`. De tag is een herkenningslabel; je deelnemerscode en servernaam blijven dezelfde als eerder. [Uitleg over tags](https://docs.digitalocean.com/products/droplets/how-to/tag/).

Controleer nogmaals dat je in team **lanupro-tech** werkt. Een project is een groepering binnen het geselecteerde team; het is geen afzonderlijk factureringsaccount.

![Aantal, naam, project en tags. Vul één server, starten-met-iot en je eigen voornaam in; de voorbeeldwaarden zijn geen practicuminstellingen.](../../afbeeldingen/schermafbeeldingen/03_naam_project_tags.png)

*Figuur 7. Aantal, naam, project en tags. Vul één server, starten-met-iot en je eigen voornaam in; de voorbeeldwaarden zijn geen practicuminstellingen. Officiële DigitalOcean-bronafbeelding, gecontroleerd op 17 november 2026. De keuzes voor deze oefening staan in de tekst erboven. [Bron en rechten](../../afbeeldingen/schermafbeeldingen/BRONNEN.md).*

### 4.8 Laatste controle en aanmaken

Loop vóór je klikt deze korte controle na:

- **lanupro-tech → starten-met-iot**;
- **Debian 13 x64**;
- **Basic → Regular → laagste prijs**, momenteel $4 per maand;
- **Password**, met het root-wachtwoord veilig bewaard;
- publiek **IPv4 aan**;
- extra opslag, automatische back-ups, monitoring, startup scripts en managed database **uit**;
- **één** server, met jouw servernaam en **voornaam als tag**.

Controleer het totaalbedrag. Een hoger bedrag dan het gekozen basisplan wijst mogelijk op een extra optie. Klik daarna één keer op **Create Droplet**. Wacht tot de aanmaak gereed is, open jouw server en kopieer het **Public IPv4**-adres. Dat vul je in bij de volgende codecel.

<div style="background-color:#fff8e1;color:#6d4c00;border-left:5px solid #f9a825;padding:14px;margin:12px 0;border-radius:4px;"><strong>Kosten</strong><br>Een Droplet wordt niet automatisch na een maand verwijderd. Ook een uitgeschakelde Droplet kan kosten blijven veroorzaken doordat de resources gereserveerd blijven. De docent beheert de einddatum en verwijdert de oefenservers na het veilig bewaren van de benodigde resultaten. Controleer de actuele tarieven via <a href="https://docs.digitalocean.com/products/droplets/details/pricing/">DigitalOcean</a>.</div>

**Uitvoeren:** in Codespaces (Notebook 03)

Vul hieronder het publieke IPv4-adres van **jouw eigen** Droplet in wanneer de invoervraag verschijnt. De code bewaart het adres in `gegevens/werk/server.json`. Die map wordt niet naar GitHub gepusht. Bij hervatten gebruikt de cel het eerder bewaarde adres.

In [3]:
serverpad = werkmap / "server.json"
if serverpad.exists():
    serverinstellingen = json.loads(serverpad.read_text(encoding="utf-8"))
    if serverinstellingen.get("deelnemerscode") != deelnemer:
        raise ValueError("Dit serverbestand hoort bij een andere deelnemer. Controleer je werkmap.")
    serveradres = str(ipaddress.IPv4Address(serverinstellingen["ipv4"]))
else:
    serveradres = str(ipaddress.IPv4Address(input("Publiek IPv4-adres van jouw Droplet: ").strip()))
    serverinstellingen = {"ipv4": serveradres, "deelnemerscode": deelnemer}
    serverpad.write_text(json.dumps(serverinstellingen, indent=2) + "\n", encoding="utf-8")
print("Controleer dit adres met jouw Droplet:", serveradres)
print("Eerste SSH-opdracht:", f"ssh -o HostKeyAlgorithms=ssh-ed25519 root@{serveradres}")

Controleer dit adres met jouw Droplet: 127.0.0.1
Eerste SSH-opdracht: ssh -o HostKeyAlgorithms=ssh-ed25519 root@127.0.0.1


## 5. Je eerste beheerverbinding

**SSH** is het protocol voor een versleutelde beheerverbinding. Je wachtwoord is de manier waarop je aantoont wie je bent. Wachtwoordaanmelding is dus ook een SSH-verbinding. Een SSH-sleutel zou een andere aanmeldmethode zijn; voor deze oefening gebruik je het wachtwoord.

Bij de eerste verbinding moet je ook vaststellen dat je de juiste server bereikt. SSH toont daarvoor een **vingerafdruk** van de serversleutel. Je vergelijkt die met de vingerafdruk die je via DigitalOcean op de server opvraagt.

**Uitvoeren:** op de DigitalOcean-website (Web Console van jouw Droplet)

Open je eigen Droplet en klik op **Web Console**. Controleer de servernaam. Voer in de console uit:

```bash
whoami
hostname
ssh-keygen -lf /etc/ssh/ssh_host_ed25519_key.pub
```

De opdracht met `ssh-keygen -lf` leest alleen de vingerafdruk van de al bestaande serversleutel. Ze maakt geen gebruikerssleutel aan in je Codespace.

Je verwacht `root`, je gekozen servernaam en een regel met `SHA256:...`. Bewaar die regel om dadelijk te vergelijken; dit is een openbare vingerafdruk, geen wachtwoord.

![Serveroverzicht met Web Console](../../afbeeldingen/schermafbeeldingen/03_serveroverzicht.png)

*Figuur 8. Serveroverzicht uit de officiële DigitalOcean-documentatie. De afbeelding toont de knop Web Console en het veld Public IPv4. Naam, adres, besturingssysteem en prijs in dit voorbeeld horen niet bij jouw server. [Bron](https://docs.digitalocean.com/products/droplets/how-to/connect-with-console/).*

**Uitvoeren:** in Codespaces (Terminal)

Kies bovenaan **Terminal → New Terminal**. Een terminal is een tekstvenster waarin je opdrachten aan een computer geeft. Kopieer de volledige SSH-opdracht uit de vorige codecel naar deze terminal en druk op Enter. Vergelijk de getoonde `SHA256:...` met de console. Typ alleen bij overeenkomst `yes`. Voer daarna je root-wachtwoord in. Bij wachtwoordinvoer zie je geen letters of sterretjes; dat is normaal.

Na het aanmelden werkt deze terminal op je **DigitalOcean-server**. Voer opnieuw `whoami` en `hostname` uit. Je verwacht dezelfde gebruiker en servernaam als in de Web Console. Met `exit` keer je later terug naar Codespaces.

<div style="background-color:#e3f2fd;color:#0d47a1;border-left:5px solid #1976d2;padding:14px;margin:12px 0;border-radius:4px;"><strong>Tip: als aanmelden niet lukt</strong><br>Gebruik de Web Console om gebruikers of instellingen na te kijken. Als SSH of de netwerkconfiguratie zelf defect is, gebruik dan de <a href="https://docs.digitalocean.com/products/droplets/how-to/recovery/recovery-console/">Recovery Console</a> via DigitalOcean. Die is een afzonderlijke herstelroute en vereist een serverwachtwoord. Verwijder een waarschuwing over een gewijzigde serversleutel niet zonder eerst opnieuw de vingerafdruk te controleren.</div>

**Vraag 2.** Een partner wil straks je MQTT-berichten lezen. Heeft die partner je root- of beheerderswachtwoord nodig? Leg ook uit waarom aanmelden met een wachtwoord nog steeds SSH gebruikt.

**Jouw antwoord (Vraag 2):**

Voeg hier je antwoord in. Onderscheid het protocol, de aanmeldmethode en het soort account.

**Antwoord 2**

SSH versleutelt de beheerverbinding; het wachtwoord authenticeert de gebruiker. Een MQTT-partner krijgt een afzonderlijk leesaccount voor de broker. Dat account geeft geen toegang tot de beheerterminal en heeft het root- of beheerderswachtwoord niet nodig.

## 6. Een gebruiker voor dagelijks beheer maken

**Uitvoeren:** op je DigitalOcean-server (SSH-terminal, als root)

`root` kan alle systeembestanden wijzigen. Je maakt daarom de benoemde gebruiker **beheer**. Met `sudo` kan die gebruiker gericht een opdracht met beheerdersrechten uitvoeren. Je ziet zo bij welke stappen extra rechten nodig zijn.

Werk eerst de lijst met beschikbare pakketten bij, installeer beveiligingsupdates en voeg de gebruiker toe. Voer ieder blok afzonderlijk uit en controleer dat het zonder fout eindigt.

```bash
apt-get update
apt-get upgrade -y
apt-get install -y sudo
```

```bash
adduser beheer
usermod -aG sudo beheer
id beheer
```

`adduser` vraagt een nieuw, afzonderlijk sterk wachtwoord. Bewaar dit. Bij de aanvullende naamvelden mag je Enter gebruiken om ze leeg te laten. Bij `id beheer` moet de groep `sudo` staan. Bestaat `beheer` al omdat je verderwerkt, sla `adduser` over en controleer `id beheer`.

**Uitvoeren:** in Codespaces (een tweede Terminal)

Laat de eerste SSH-terminal open. Open met **Terminal → New Terminal** een tweede terminal en gebruik de volgende codecel om de opdracht voor je eigen adres te tonen.

In [4]:
print(f"ssh -o HostKeyAlgorithms=ssh-ed25519 beheer@{serveradres}")

ssh -o HostKeyAlgorithms=ssh-ed25519 beheer@127.0.0.1


Kopieer die opdracht naar de tweede terminal. Meld je aan met het wachtwoord van **beheer**. Voer daar uit:

```bash
whoami
sudo -v
sudo whoami
```

De eerste opdracht geeft `beheer`; de laatste geeft `root`. `sudo -v` vraagt zo nodig opnieuw het wachtwoord van **beheer**. Ga pas verder wanneer beide aanmeldingen werken. Sluit dan de oude root-terminal met `exit`.

**Uitvoeren:** op je DigitalOcean-server (SSH-terminal, als beheer)

Maak direct aanmelden als root via SSH onmogelijk, terwijl wachtwoordaanmelding voor beheer beschikbaar blijft. `tee` schrijft de regels tussen de twee `EOF`-markeringen naar een configuratiebestand.

```bash
sudo tee /etc/ssh/sshd_config.d/00-practicum.conf >/dev/null <<'EOF'
PasswordAuthentication yes
PermitRootLogin no
MaxAuthTries 3
EOF
sudo /usr/sbin/sshd -t
sudo /usr/sbin/sshd -T | grep -E '^(passwordauthentication|permitrootlogin|maxauthtries) '
```

Je verwacht `passwordauthentication yes`, `permitrootlogin no` en `maxauthtries 3`. Als de controle afwijkt of een fout geeft, herstel het bestand vóór je herlaadt. Bij een correcte controle:

```bash
sudo systemctl reload ssh
```

Houd je huidige verbinding open en meld je in een nieuwe terminal nogmaals aan als **beheer**. Zo controleer je dat de nieuwe instellingen werken voordat je een werkende sessie sluit.

## 7. De programma's installeren

**Uitvoeren:** op je DigitalOcean-server (SSH-terminal, als beheer)

Een **pakketbeheerder** haalt software uit de pakketbronnen van Debian en installeert de bijbehorende onderdelen. Je installeert Mosquitto voor MQTT, Python voor onze programma's, SQLite-hulpmiddelen om databanken te onderzoeken, Git voor versiebeheer en Nginx voor de beveiligde webverbinding. OpenSSL maakt certificaten; UFW beheert de firewallregels.

```bash
sudo apt-get install -y mosquitto mosquitto-clients python3 python3-venv sqlite3 git openssl ca-certificates nginx ufw
sudo systemctl disable --now mosquitto
```

Controleer de geïnstalleerde software en het besturingssysteem:

```bash
cat /etc/os-release
python3 --version
mosquitto -h | head -n 2
sqlite3 --version
git --version
```

Je verwacht Debian 13, Python 3.13 en Mosquitto 2.0.x uit de Debian-pakketbron. Het laatste versienummer kan door beveiligingsupdates verschillen. Gebruik geen opdracht om met `pip` de systeem-Python te overschrijven.

Een **proces** is een programma dat momenteel draait. Een **service** laat je dat programma met `systemctl` starten en stoppen. Debian kan Mosquitto bij de installatie al starten. Met `disable --now` hierboven schakel je automatisch starten bij het opstarten van de server uit en stop je het huidige brokerproces. Je start de broker later zelf, zodra de configuratie klaar is. Controleer de toestand:

```bash
sudo systemctl status mosquitto --no-pager
```

Je verwacht `inactive (dead)` en bij de service-instellingen `disabled`. Dat is hier de bedoelde toestand. De installatie is aanwezig, maar de broker draait nog niet. `systemctl status` kan bij een gestopte service een niet-nul afsluitcode geven; die betekent hier niet dat de installatie mislukt is.

**Vraag 3.** Zoek de werkelijk geïnstalleerde Python- en Mosquitto-versie op en noteer de toestand van de Mosquitto-service. Wat bewijst active (running) wel en niet?

**Jouw antwoord (Vraag 3):**

Voeg hier je antwoord in. Noteer je eigen uitvoer en leg uit waarom later nog een verbindingstest nodig is.

**Antwoord 3**

De exacte versienummers volgen uit de eigen terminaluitvoer. Bij Debian 13 is Python 3.13 en Mosquitto 2.0.x het verwachte hoofdpatroon. active (running) betekent dat het serviceproces draait; het bewijst nog niet dat de firewall, certificaten, wachtwoorden en topicrechten een externe client toelaten.

## 8. De meegeleverde serverbestanden overbrengen

**Uitvoeren:** in Codespaces (Terminal, zonder SSH-verbinding)

De map `server/` bevat de korte programma's en configuratievoorbeelden voor deze installatie. Met **SCP** kopieer je die bestanden over dezelfde versleutelde beheerverbinding. Kopiëren voert de bestanden nog niet uit.

Open een nieuwe terminal in Codespaces. Ga naar de hoofdmap van de repository en voer uit:

```bash
cd "$(git rev-parse --show-toplevel)"
ls server
```

Je verwacht onder meer `certificaat_maken.py`, `toegang_api.py`, `statusdienst.py`, `requirements.txt`, `configuratie/` en `diensten/`. Ontbreekt de map, dan heeft je branch de bestanden van notebook 03 nog niet. Bewaar je werk en laat je branch bijwerken voordat je doorgaat.

De volgende cel toont de kopieeropdracht. Voer die in deze Codespaces-terminal uit en vul het wachtwoord van **beheer** in wanneer SCP erom vraagt.

In [5]:
print(f"scp -o HostKeyAlgorithms=ssh-ed25519 -r server beheer@{serveradres}:~/practicum-server")

scp -o HostKeyAlgorithms=ssh-ed25519 -r server beheer@127.0.0.1:~/practicum-server


<div style="background-color:#e3f2fd;color:#0d47a1;border-left:5px solid #1976d2;padding:14px;margin:12px 0;border-radius:4px;"><strong>Tip: opnieuw kopiëren</strong><br>Kopieer de map met deze opdracht alleen als <code>~/practicum-server</code> nog niet bestaat. Bij hervatten controleer je eerst de bestaande map. Voor het bewust bijwerken van bestanden gebruik je <code>scp -r server/. beheer@JOUW_IP:~/practicum-server/</code>, met je eigen IPv4-adres in plaats van JOUW_IP. Daarmee ontstaat geen extra geneste map.</div>

**Uitvoeren:** op je DigitalOcean-server (SSH-terminal, als beheer)

Controleer dat de bestanden op de server aangekomen zijn:

```bash
ls ~/practicum-server
```

Maak een aparte servicegebruiker **iot**. Dit is geen extra persoon met een wachtwoord: het account draait alleen onze webdienst. De programma's komen in `/opt/iot`; eventuele latere servergegevens komen in `/var/lib/iot`.

```bash
id iot >/dev/null 2>&1 || sudo useradd --system --user-group --home-dir /var/lib/iot --create-home --shell /usr/sbin/nologin iot
sudo install -d -m 755 /opt/iot/programmatuur /etc/iot
sudo python3 -m venv /opt/iot/venv
sudo /opt/iot/venv/bin/python -m pip install -r ~/practicum-server/requirements.txt
sudo install -m 644 ~/practicum-server/statusdienst.py ~/practicum-server/toegang_api.py /opt/iot/programmatuur/
sudo /opt/iot/venv/bin/python -m pip check
```

Een **virtuele Python-omgeving** bevat de pakketten voor deze toepassing, gescheiden van de Python-onderdelen van Debian. `pip check` hoort geen conflicten te melden. De meegeleverde versies zijn samen gecontroleerd.

FastAPI beschrijft het webantwoord; Uvicorn voert dat Python-programma uit. We beginnen met één beveiligd controleantwoord. De ontvanger voor meetrecords en de gegevensroutes worden in notebooks 05 en 06 toegevoegd aan deze serveromgeving.

## 9. Versleuteling en de identiteit van je server

Een wachtwoord beschermt de aanmelding, maar zonder versleuteling kan het onderweg leesbaar zijn. **TLS** versleutelt de verbinding en laat de client het certificaat van de server controleren. HTTPS is HTTP over TLS; MQTT gebruikt hier eveneens TLS.

Voor deze oefening gebruiken we het vaste publieke IPv4-adres van je Droplet. Je maakt een kleine eigen **certificaatautoriteit** (CA) die het servercertificaat ondertekent. Codespaces vertrouwt die CA straks alleen voor de verbindingen in deze oefening. Je hoeft daarvoor geen domeinnaam te kopen of de beveiliging van je browser te veranderen.

| Bestand | Betekenis | Waar blijft het? |
|---|---|---|
| `ca.crt` | Openbaar certificaat waarmee je de ondertekening controleert | Server en Codespaces; later ook bij je gegevenspartner |
| `ca.key` | Privésleutel waarmee de CA ondertekent | Alleen op de server, toegankelijk voor root |
| `server.crt` | Certificaat met het IPv4-adres van deze server | Op de server; de server stuurt het bij de verbinding mee |
| `server.key` | Privésleutel van de server | Alleen op de server, met beperkte leesrechten |

**Uitvoeren:** op je DigitalOcean-server (SSH-terminal, als beheer)

Vul bij de twee vragen je publieke IPv4-adres en de deelnemerscode uit de eerste notebookcel in. Dit zijn geen geheimen.

```bash
read -rp 'Publiek IPv4-adres van deze Droplet: ' SERVER_IP
read -rp 'Deelnemerscode uit notebook 01: ' DEELNEMER
sudo python3 ~/practicum-server/certificaat_maken.py --adres "$SERVER_IP" --deelnemer "$DEELNEMER"
sudo openssl x509 -in /etc/iot/certificaten/server.crt -noout -dates -ext subjectAltName
```

Controleer dat `IP Address:` jouw publieke IPv4-adres noemt. Het servercertificaat is 90 dagen geldig. De hulpcode weigert een bestaande certificaatmap voor een ander adres te vervangen. Bij opnieuw uitvoeren met dezelfde instellingen blijven de bestaande certificaten behouden.

Maak de openbare CA beschikbaar om die via SCP op te halen. Geef Mosquitto een eigen kopie van het servercertificaat en zijn privésleutel. Alleen root en de Mosquitto-service mogen die sleutel lezen.

```bash
sudo install -m 644 /etc/iot/certificaten/ca.crt /etc/iot/ca.crt
sudo install -o root -g mosquitto -m 640 /etc/iot/certificaten/server.crt /etc/mosquitto/certs/server.crt
sudo install -o root -g mosquitto -m 640 /etc/iot/certificaten/server.key /etc/mosquitto/certs/server.key
sudo openssl verify -CAfile /etc/iot/ca.crt -verify_ip "$SERVER_IP" /etc/iot/certificaten/server.crt
```

De laatste regel hoort te eindigen op `OK`. Bewaar de privésleutels nooit in je repository. Als je later de Droplet vervangt en een ander IP-adres krijgt, moet de certificaatinrichting opnieuw gecontroleerd worden.

**Vraag 4.** Waarom kun je ca.crt wel naar Codespaces kopiëren, terwijl server.key op de server moet blijven? Waarom controleer je ook het IP-adres in het certificaat?

**Jouw antwoord (Vraag 4):**

Voeg hier je antwoord in. Leg het verschil uit tussen openbare controle-informatie en een privésleutel.

**Antwoord 4**

ca.crt bevat openbare informatie om te controleren welke CA het certificaat heeft ondertekend. server.key bewijst namens de server zijn identiteit en moet geheim blijven. Ook een geldig ondertekend certificaat moet bij het bedoelde IP-adres horen; anders kan het certificaat van een andere server zijn.

## 10. De MQTT-broker toegankelijk maken

Een **MQTT-client** maakt verbinding met de broker om een bericht te publiceren of om zich op berichten te abonneren. Bij ieder bericht hoort een **topic**, een tekst die de berichtenstroom aanduidt. De broker gebruikt het topic om te bepalen welke abonnees het bericht ontvangen.

Je eigen broker krijgt twee accounts. **eigenaar** mag berichten schrijven en lezen onder `practicum/jouw_deelnemerscode/`. **lezer** mag alleen de meet- en controleberichten onder diezelfde deelnemerscode lezen. Een partner hoeft daardoor geen beheerrechten of schrijfrechten te krijgen.

**Uitvoeren:** op je DigitalOcean-server (SSH-terminal, als beheer)

Maak eerst het wachtwoordbestand en geef beide accounts een verschillend, sterk wachtwoord. De invoer gebeurt verborgen in de terminal. Gebruik andere wachtwoorden dan voor SSH.

```bash
sudo test -e /etc/mosquitto/practicum.wachtwoorden || sudo install -o root -g mosquitto -m 640 /dev/null /etc/mosquitto/practicum.wachtwoorden
sudo mosquitto_passwd /etc/mosquitto/practicum.wachtwoorden eigenaar
sudo mosquitto_passwd /etc/mosquitto/practicum.wachtwoorden lezer
sudo chown root:mosquitto /etc/mosquitto/practicum.wachtwoorden
sudo chmod 640 /etc/mosquitto/practicum.wachtwoorden
```

Bewaar de twee wachtwoorden met hun accountnaam in je wachtwoordmanager. Bij hervatten hoef je ze niet opnieuw in te stellen. Voer je een van de `mosquitto_passwd`-opdrachten opnieuw uit, dan wijzig je dat accountwachtwoord.

Maak nu de topicrechten. Vul de deelnemerscode opnieuw in wanneer erom gevraagd wordt; zo werkt dit ook na een nieuwe SSH-aanmelding.

```bash
read -rp 'Deelnemerscode uit notebook 01: ' DEELNEMER
if [[ "$DEELNEMER" =~ ^[a-z0-9][a-z0-9_-]{0,62}$ ]]; then
    printf 'user eigenaar
topic readwrite practicum/%s/#

user lezer
topic read practicum/%s/metingen/#
topic read practicum/%s/controle/#
' "$DEELNEMER" "$DEELNEMER" "$DEELNEMER" | sudo tee /etc/mosquitto/practicum.rechten >/dev/null
else
    echo 'Ongeldige deelnemerscode. Herhaal dit blok met de code uit notebook 01.'
fi
sudo chown root:mosquitto /etc/mosquitto/practicum.rechten
sudo chmod 640 /etc/mosquitto/practicum.rechten
sudo cat /etc/mosquitto/practicum.rechten
```

Controleer de deelnemerscode in alle drie topicregels. Het teken `#` omvat onderliggende topics. Het rechtenbestand bevat geen wachtwoorden en mag je bekijken. Installeer vervolgens de brokerconfiguratie:

```bash
sudo install -m 644 ~/practicum-server/configuratie/mosquitto.conf /etc/mosquitto/conf.d/practicum.conf
sudo systemctl disable mosquitto
sudo systemctl start mosquitto
sudo systemctl status mosquitto --no-pager
```

Je verwacht nu `active (running)`. `start` start de broker voor deze werksessie; `disabled` zorgt dat hij na een serverherstart uit blijft. Bij latere wijzigingen aan de brokerconfiguratie gebruik je bewust `sudo systemctl restart mosquitto`. Mosquitto luistert nu op poort **8883**, vereist een account en gebruikt het certificaat. Een fout bij herstarten los je eerst op met het service-logboek in de herstelparagraaf.

## 11. De beveiligde webverbinding klaarzetten

**Uitvoeren:** op je DigitalOcean-server (SSH-terminal, als beheer)

Een API laat een programma gericht gegevens aan een ander programma vragen. Om de verbinding alvast te controleren, installeren we een klein antwoord op `/gezondheid`. **Nginx** ontvangt HTTPS op poort 443 en geeft de aanvraag binnen de server door aan Uvicorn op poort 8000. Die interne poort is alleen bereikbaar op `127.0.0.1`, het adres waarmee een computer zichzelf aanspreekt.

Stel een afzonderlijk API-wachtwoord in. Gebruik minstens twintig willekeurige tekens uit je wachtwoordmanager, zonder accenten of andere niet-ASCII-tekens. Het programma bewaart alleen een afgeleide controlewaarde; het wachtwoord verschijnt niet in de uitvoer.

```bash
sudo python3 ~/practicum-server/toegang_api.py
sudo chown root:iot /etc/iot/api.controlewaarde
sudo chmod 640 /etc/iot/api.controlewaarde
```

Bij hervatten meldt het programma dat de bestaande toegang behouden blijft. Dat is geen reden om het bestand te verwijderen. Gebruik dan je eerder bewaarde API-wachtwoord.

Installeer de service en de HTTPS-configuratie. Het verwijderen van `sites-enabled/default` schakelt alleen de standaard welkomstpagina van deze nieuwe oefenserver uit.

```bash
sudo install -m 644 ~/practicum-server/diensten/iot-status.service /etc/systemd/system/iot-status.service
sudo systemctl daemon-reload
sudo systemctl enable --now iot-status
sudo install -m 644 ~/practicum-server/configuratie/nginx.conf /etc/nginx/sites-available/practicum
sudo ln -sfn /etc/nginx/sites-available/practicum /etc/nginx/sites-enabled/practicum
sudo rm -f /etc/nginx/sites-enabled/default
sudo nginx -t
```

Ga alleen verder wanneer Nginx `syntax is ok` en `test is successful` meldt:

```bash
sudo systemctl restart nginx
sudo systemctl enable nginx
sudo systemctl is-active mosquitto iot-status nginx
sudo systemctl is-enabled mosquitto iot-status nginx
```

Je verwacht driemaal `active`. Bij `is-enabled` verwacht je, in dezelfde volgorde, `disabled`, `enabled`, `enabled`: Mosquitto start je zelf na een serverherstart. De webdiensten `iot-status` en Nginx starten wel automatisch. Het sluiten van je SSH-terminal stopt een eenmaal gestarte broker niet.

## 12. Alleen de benodigde poorten openen

**Uitvoeren:** op je DigitalOcean-server (SSH-terminal, als beheer)

Een **firewall** bepaalt welk netwerkverkeer de server mag ontvangen. We laten SSH, beveiligd MQTT en HTTPS toe. De interne webdienst op 8000 en onbeveiligd MQTT op 1883 krijgen geen externe toegang.

Houd je huidige SSH-terminal open. Sta poort 22 toe **vóór** je de firewall inschakelt, zodat je de beheerverbinding behoudt.

```bash
sudo ufw default deny incoming
sudo ufw default allow outgoing
sudo ufw limit 22/tcp
sudo ufw allow 8883/tcp
sudo ufw allow 443/tcp
sudo ufw enable
sudo ufw status numbered
sudo ss -lntp
```

Bevestig de UFW-vraag met `y`. `limit` beperkt herhaalde SSH-verbindingspogingen vanaf hetzelfde adres. Controleer dat 22, 8883 en 443 in de firewallregels staan. Bij de luisterende programma's verwacht je 8883 en 443 extern, en 8000 alleen op `127.0.0.1`.

Open een nieuwe Codespaces-terminal en meld je opnieuw aan als **beheer**. Sluit je bestaande verbinding pas wanneer dat werkt.

<div style="background-color:#e3f2fd;color:#0d47a1;border-left:5px solid #1976d2;padding:14px;margin:12px 0;border-radius:4px;"><strong>Tip: een tweede firewall bij DigitalOcean</strong><br>Als het team ook een DigitalOcean Cloud Firewall aan je Droplet koppelt, moet die eveneens TCP 22, 8883 en 443 toelaten vanaf de afgesproken clientadressen. Codespaces-adressen kunnen veranderen. Laat de docent deze centrale regels controleren; verander geen firewall die ook servers van anderen beschermt.</div>

## 13. De verbinding vanuit Codespaces testen

**Uitvoeren:** in Codespaces (Terminal, zonder SSH-verbinding)

Je haalt het openbare CA-certificaat op via de SSH-verbinding waarvan je de vingerafdruk hebt gecontroleerd. Zo leg je vast welke servercertificaten je voor deze oefening vertrouwt. Het bestand is openbaar, maar de herkomst moet kloppen.

Voer de opdracht uit die deze cel toont, vanuit de hoofdmap van je repository. Gebruik het wachtwoord van **beheer**.

In [6]:
print(f"scp -o HostKeyAlgorithms=ssh-ed25519 beheer@{serveradres}:/etc/iot/ca.crt gegevens/werk/ca.crt")

scp -o HostKeyAlgorithms=ssh-ed25519 beheer@127.0.0.1:/etc/iot/ca.crt gegevens/werk/ca.crt


**Uitvoeren:** in Codespaces (Notebook 03)

De volgende cel vraagt eerst het MQTT-wachtwoord van **eigenaar** en daarna je afzonderlijke API-wachtwoord. De invoer verschijnt verborgen. Vul geen SSH-wachtwoord in.

De controle meldt zich aan bij je broker, wacht op de abonnementsbevestiging en stuurt dan een tijdelijk controlebericht. Alleen wanneer hetzelfde bericht terugkomt, slaagt die stap. Vervolgens vraagt de code via HTTPS het beveiligde controleantwoord op. De code controleert het certificaat en het serveradres bij beide verbindingen.

Het controlebericht is geen biggenmeting. Het komt op een apart controletopic en wordt niet als bewaard MQTT-bericht gepubliceerd. De ingevulde wachtwoorden worden niet in bestanden of celuitvoer bewaard.

In [7]:
from getpass import getpass
from datetime import datetime, timezone

sys.path.insert(0, str(hoofdmap / "programmatuur"))
from servercontrole import controleer

verbindingsresultaat = {}
mqtt_wachtwoord = api_wachtwoord = None
try:
    mqtt_wachtwoord = getpass("MQTT-wachtwoord van eigenaar: ")
    api_wachtwoord = getpass("Afzonderlijk API-wachtwoord: ")
    verbindingsresultaat = controleer(serveradres, deelnemer, werkmap / "ca.crt", mqtt_wachtwoord, api_wachtwoord)
finally:
    del mqtt_wachtwoord, api_wachtwoord
print(json.dumps(verbindingsresultaat, indent=2))

{
  "mqtt_tls": "geslaagd",
  "mqtt_heen_en_terug": "geslaagd",
  "https": "geslaagd",
  "api_aanmelding": "geslaagd"
}


<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;border-radius:4px;"><strong>Controle</strong><br>Je verwacht vier keer <code>geslaagd</code>. Daarmee heb je vanaf deze werkplek aangetoond dat MQTT-aanmelding, publiceren en ontvangen, certificaatcontrole en de beveiligde webaanmelding werken. De verwerking en opslag van echte meetrecords volgen in de volgende notebooks.</div>

**Vraag 5.** Sluit alleen je SSH-terminal met exit en voer de verbindingstest opnieuw uit. Werkt die nog? Verklaar de uitkomst.

**Jouw antwoord (Vraag 5):**

Voeg hier je antwoord in. Maak onderscheid tussen je beheersessie en de services op de Droplet.

**Antwoord 5**

De verbindingstest hoort opnieuw te slagen. exit sluit de SSH-beheersessie. De services Mosquitto, iot-status en Nginx worden door het besturingssysteem beheerd en blijven op de Droplet actief.

## 14. Zelf starten na een serverherstart

**Uitvoeren:** op je DigitalOcean-server (SSH-terminal, als beheer)

Na een serverherstart blijft de broker uit totdat je hem zelf start. Je controleert dit op je eigen Droplet. Voer de oefening uit voordat een partner de broker gebruikt.

```bash
sudo reboot
```

Je SSH-verbinding wordt verbroken. Wacht totdat de Droplet weer actief is in DigitalOcean en meld je opnieuw aan als **beheer**. Controleer eerst, voordat je iets start:

```bash
sudo systemctl is-active mosquitto
sudo systemctl is-enabled mosquitto
sudo systemctl is-active iot-status nginx
sudo ufw status
```

Je verwacht voor Mosquitto `inactive` en `disabled`. De webdiensten horen wel `active` te zijn. Een niet-nul afsluitcode bij de eerste twee opdrachten is hier normaal: de broker staat bewust uit. Start hem nu zelf:

```bash
sudo systemctl start mosquitto
sudo systemctl status mosquitto --no-pager
```

Je verwacht `active (running)`. Gebruik deze startopdracht telkens na een serverherstart, vóór je berichten verstuurt of je partner laat verbinden. Wil je de broker bewust stoppen, gebruik dan `sudo systemctl stop mosquitto`. De installatie, wachtwoorden en certificaten blijven daarbij bewaard.

**Uitvoeren:** in Codespaces (Notebook 03)

Voer na het handmatig starten de verbindingstest opnieuw uit. Noteer hieronder de toestand vóór en na het starten en de uitkomst van de test.

**Vraag 6.** Welke toestand toont Mosquitto direct na de serverherstart en na jouw startopdracht? Wat geeft de verbindingstest daarna? Verklaar waarom `disabled` en `active` tegelijk kunnen voorkomen.

**Jouw antwoord (Vraag 6):**

Voeg hier je antwoord in. Noteer beide toestanden en onderscheid automatisch starten van momenteel actief zijn.

**Antwoord 6**

Direct na de serverherstart is Mosquitto `inactive` en `disabled`. Na `sudo systemctl start mosquitto` is hij `active (running)`, maar nog steeds `disabled`. De eerste toestand beschrijft of het programma nu draait; de tweede of het automatisch start bij het opstarten van de server. Na de handmatige start horen de vier verbindingcontroles te slagen. De configuratie is bewaard gebleven.

## 15. Je resultaat bewaren en afsluiten

**Uitvoeren:** in Codespaces (Notebook 03)

Bewaar een klein controlebestand zodat je later ziet welke verbindingen je hebt getest. Het bestand bevat je deelnemerscode, testtijd en uitkomsten. Wachtwoorden, privésleutels en je serveradres horen niet in dit inleverbestand.

In [8]:
if not all(verbindingsresultaat.get(k) == "geslaagd" for k in ("mqtt_tls", "mqtt_heen_en_terug", "https", "api_aanmelding")):
    raise RuntimeError("Los eerst de mislukte verbindingstest op.")
verslag = {"deelnemerscode": deelnemer, "branch": branch,
           "getest_op_utc": datetime.now(timezone.utc).isoformat(),
           "verbindingen": verbindingsresultaat}
inleverpad = hoofdmap / "gegevens/inlevering/03_servercontrole.json"
inleverpad.parent.mkdir(parents=True, exist_ok=True)
inleverpad.write_text(json.dumps(verslag, indent=2) + "\n", encoding="utf-8")
print("Bewaard:", inleverpad.relative_to(hoofdmap))
print("Bewaar je eigen antwoorden over de service- en serverherstart in het notebook.")

Bewaard: gegevens/inlevering/03_servercontrole.json
Bewaar je eigen antwoorden over de service- en serverherstart in het notebook.


<div style="background-color:#fff8e1;color:#6d4c00;border-left:5px solid #f9a825;padding:14px;margin:12px 0;border-radius:4px;"><strong>Stop je Codespace na het practicum</strong><br>Stop de Codespace na het bewaren van je werk om onnodig gebruik van de beschikbare uren te vermijden. De DigitalOcean-Droplet heeft een afzonderlijke facturering en blijft beschikbaar voor de vervolgopdrachten. Verwijder die pas volgens de afspraak met de docent.</div>

**Uitvoeren:** in Codespaces (Source Control) en op de GitHub-website

1. Vul je eigen antwoorden aan. Gebruik **File → Save All** vóór je wijzigingen selecteert.
2. Controleer de notebookuitvoer. Verwijder eventuele onverwacht zichtbare geheimen vóór een commit. De tijdelijke uitvoer met je serveradres kun je leegmaken; het adres blijft in `gegevens/werk/server.json` bewaard.
3. Selecteer uitsluitend je aangepaste notebook en `gegevens/inlevering/03_servercontrole.json`. Gebruik een duidelijke commitboodschap en **Push** naar je eigen branch.
4. Controleer op GitHub je branch en de opgeslagen bestanden.
5. Bewaar je servernaam en wachtwoorden in je wachtwoordmanager. `server.json` en `ca.crt` blijven in de bestaande Codespace staan; bij een nieuwe Codespace vul je het adres opnieuw in en haal je het openbare certificaat opnieuw op via gecontroleerde SSH.
6. Sluit de SSH-terminal met `exit`. Stop daarna de Codespace via [github.com/codespaces](https://github.com/codespaces). Alleen het browsertabblad sluiten is onvoldoende.

Voor de volgende sessie hervat je dezelfde Codespace en dezelfde Droplet. Is de server intussen herstart, meld je dan aan als beheer en voer `sudo systemctl start mosquitto` uit vóór de verbindingstest. Alleen je Codespace stoppen of je SSH-terminal sluiten stopt de broker niet. Het opnieuw aanmaken van de omgevingen is niet nodig.

## Hulp bij foutmeldingen

| Waarneming | Controle en herstel |
|---|---|
| Team of project ontbreekt | Controleer je account, teamuitnodiging en geselecteerde team; laat de docent je rechten nakijken. |
| SSH geeft Permission denied | Gebruik na de eerste inrichting `beheer` en diens wachtwoord. Directe root-aanmelding is uitgeschakeld. |
| SSH geeft een gewijzigde host key | Vergelijk opnieuw met de vingerafdruk in de DigitalOcean-console. Accepteer geen vervanging op basis van alleen de foutmelding. |
| Verbinding krijgt een time-out | Controleer of je eigen Droplet draait en of het IPv4-adres, UFW en eventuele Cloud Firewall kloppen. |
| Certificaatcontrole mislukt | Controleer serveradres, datum/tijd, vervaldatum en de via SSH opgehaalde `ca.crt`. Schakel certificaatcontrole niet uit. |
| MQTT geeft Connection refused na een serverherstart | Start de broker op de server met `sudo systemctl start mosquitto` en controleer zijn status. |
| MQTT-aanmelding geweigerd | Gebruik het MQTT-account `eigenaar` en het bijbehorende wachtwoord, niet het SSH- of API-wachtwoord. |
| Controlebericht komt niet terug | Controleer de deelnemerscode in `practicum.rechten` en notebook 01. Herstart Mosquitto na het herstellen van zijn configuratie. |
| HTTPS geeft 401 | De verbinding werkt, maar de API-aanmelding is ongeldig. Gebruik het afzonderlijke API-wachtwoord. |
| HTTPS geeft 502 | Nginx draait, maar de interne statusdienst is niet bereikbaar. Controleer `iot-status`. |
| HTTPS geeft 429 | Er kwamen te veel aanvragen kort na elkaar. Wacht even en voer één controle uit. |
| Certificaatmap bestaat al | Bij dezelfde instellingen blijven bestaande certificaten behouden. Bij een ander adres of onvolledige map vraag je hulp; overschrijf de sleutels niet. |

**Uitvoeren:** op je DigitalOcean-server (SSH-terminal of herstelconsole)

Lees bij een servicefout de laatste logregels. Deze opdrachten veranderen niets:

```bash
sudo journalctl -u mosquitto -n 30 --no-pager
sudo journalctl -u iot-status -n 30 --no-pager
sudo journalctl -u nginx -n 30 --no-pager
sudo tail -n 30 /var/log/mosquitto/mosquitto.log
sudo nginx -t
```

Een padfout of `Permission denied` in een service wijst vaak op een ontbrekend bestand of onjuiste leesrechten. Controleer de bijbehorende stap. Plak geen wachtwoorden, privésleutels of volledige geheime configuratiebestanden in een antwoordcel of GitHub-bericht.

Bij verlies van je API-wachtwoord laat je de docent een nieuwe controlewaarde instellen en de statusdienst herstarten. Een verlopen servercertificaat vraagt een gecontroleerde vernieuwing en herstart van de TLS-diensten. Bij vervanging van de CA moeten ook de clients hun vertrouwde certificaat opnieuw ophalen. Deze uitzonderingen zijn geen gewone hervatstappen.

## Technische bronnen

- [DigitalOcean: een Droplet aanmaken](https://docs.digitalocean.com/products/droplets/how-to/create/), [SSH](https://docs.digitalocean.com/products/droplets/how-to/connect-with-ssh/) en [Recovery Console](https://docs.digitalocean.com/products/droplets/how-to/recovery/recovery-console/).
- [Debian 13: Mosquitto-pakket](https://packages.debian.org/trixie/mosquitto) en [Python-pakket](https://packages.debian.org/trixie/python3).
- [Mosquitto: authenticatie](https://mosquitto.org/documentation/authentication-methods/) en [configuratie](https://mosquitto.org/man/mosquitto-conf-5.html). De lesconfiguratie gebruikt de Mosquitto 2.0-versie uit Debian 13; instructies voor nieuwere hoofdversies kunnen verschillen.
- [Paho: MQTT-client](https://eclipse.dev/paho/files/paho.mqtt.python/html/client.html), [Nginx: HTTPS](https://nginx.org/en/docs/http/configuring_https_servers.html) en [FastAPI: HTTP Basic](https://fastapi.tiangolo.com/advanced/security/http-basic-auth/).
- [Bronnen van de schermafbeeldingen](../../afbeeldingen/schermafbeeldingen/BRONNEN.md).